# Activation Functions Zoo: Rectified, Bounded, Gated, Periodic, and Self-Normalizing (C03 Topic 01)

### Historical Lineage & Theoretical Foundations
Nonlinear activation functions break layer-wise affine collapse across deep architectures:
- **Rectified Linear Units (Nair & Hinton 2010):** Scale-invariant sparsity $\max(0, x)$ solving vanishing gradients in deep networks.
- **Gaussian Error Linear Unit (Hendrycks & Gimpel 2016):** Probabilistic input weighting $\text{GELU}(x) = x \Phi(x) = x \cdot \frac{1}{2}[1 + \text{erf}(x/\sqrt{2})]$.
- **Self-Normalizing SELU (Klambauer et al. 2017):** Fixed-point dynamical contraction toward $(\mu=0, \sigma^2=1)$ without normalization layers.
- **Gated Linear Units (Shazeer 2020):** Multiplicative bilinear gates (SwiGLU, GeGLU) with $\frac{8}{3}d$ hidden dimension parameter-FLOP equivalence.
- **Periodic Inductive Bias (Ziyin et al. 2020):** Snake activation $x + \frac{1}{a}\sin^2(ax)$ enabling periodic extrapolation in audio and physical systems.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT use un-leaky ReLU with large learning rates (risks permanent **Dying ReLU** neuron death). Do NOT pair SELU with standard BatchNorm or standard Dropout (destroys the analytical fixed point; use **AlphaDropout**). Do NOT model audio or wave phenomena with pure ReLUs; use **Snake**.

### 🔗 Topic Connections & Interdependencies
- **Prerequisites:** [01_linear_neuron_and_invertible_transforms.ipynb](../00_foundations/01_linear_neuron_and_invertible_transforms.ipynb), [02_tree_vs_kernel_vs_neural_inductive_biases.ipynb](../01_paradigms_and_dichotomies/02_tree_vs_kernel_vs_neural_inductive_biases.ipynb).
- **Downstream Syntheses:** `05_architectures/04_transformer_blocks`, `06_smooth_and_physics_informed_learning/04_pinns`.


### Mathematical Formulations & Asymptotic Complexity
$$\text{SwiGLU}(x) = (\text{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}})) W_{\text{down}}, \quad d_{\text{ffn}} = \frac{8}{3}d$$
$$\text{Snake}(x; a) = x + \frac{1 - \cos(2ax)}{2a}, \quad \text{SELU}(x) = \lambda \cdot \begin{cases} x & x > 0 \\ \alpha(e^x - 1) & x \le 0 \end{cases}$$
$$\text{GELU}(x) \approx 0.5x(1 + \tanh(\sqrt{2/\pi}(x + 0.044715x^3))), \quad \text{LeakyReLU}(x) = \max(\alpha x, x)$$

- **Complexity:** Elementwise activations: $\mathcal{O}(N)$ FLOPs, memory-bandwidth bound; Gated FFN (SwiGLU/GeGLU): $\mathcal{O}(B \cdot d \cdot d_{\text{ffn}})$ compute-bound GEMM.


In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/01_activation_functions_zoo.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class ActivationZooModule(nn.Module):
    def __init__(self, w_gate: torch.Tensor, w_up: torch.Tensor, w_down: torch.Tensor, snake_a: float = 2.0):
        super().__init__()
        self.register_buffer("w_gate", w_gate)
        self.register_buffer("w_up", w_up)
        self.register_buffer("w_down", w_down)
        self.snake_a = snake_a
        self.selu_scale = 1.0507009873554805
        self.selu_alpha = 1.6732632423543772
        self.linear = nn.Linear(8, 4)

    def swiglu(self, x: torch.Tensor):
        return torch.matmul(F.silu(torch.matmul(x, self.w_gate)) * torch.matmul(x, self.w_up), self.w_down)

    def geglu(self, x: torch.Tensor):
        return torch.matmul(F.gelu(torch.matmul(x, self.w_gate)) * torch.matmul(x, self.w_up), self.w_down)

    def snake(self, x: torch.Tensor):
        return x + (1.0 - torch.cos(2.0 * self.snake_a * x)) / (2.0 * self.snake_a)

    def selu(self, x: torch.Tensor):
        return torch.where(x > 0.0, self.selu_scale * x, self.selu_scale * self.selu_alpha * (torch.exp(x) - 1.0))

    def forward(self, x: torch.Tensor): return self.linear(x)

model = ActivationZooModule(weights["w_gate"], weights["w_up"], weights["w_down"], float(weights["snake_a"][0])).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights["weight"].t()); model.linear.bias.copy_(weights["bias"])
print("ActivationZooModule initialized successfully.")


In [ ]:
assert torch.allclose(model(test_in), test_out, atol=1e-5), "Linear anchor mismatch"
swiglu_res = model.swiglu(weights["x_gate"])
assert torch.allclose(swiglu_res, weights["swiglu_out"], atol=1e-5), "SwiGLU mismatch"

geglu_res = model.geglu(weights["x_gate"])
assert torch.allclose(geglu_res, weights["geglu_out"], atol=1e-5), "GeGLU mismatch"

snake_res = model.snake(weights["x_snake"])
assert torch.allclose(snake_res, weights["snake_out"], atol=1e-5), "Snake mismatch"

selu_res = model.selu(weights["x_selu"])
assert torch.allclose(selu_res, weights["selu_out"], atol=1e-5), "SELU mismatch"

assert torch.allclose(F.relu(weights["x_dead"]), weights["relu_out"], atol=1e-5), "ReLU mismatch"
assert torch.allclose(F.leaky_relu(weights["x_dead"], 0.1), weights["leaky_relu_out"], atol=1e-5), "LeakyReLU mismatch"
assert torch.allclose(F.silu(weights["x_dead"]), weights["silu_out"], atol=1e-5), "SiLU mismatch"
print("All 5 activation families verified with zero error!")


In [ ]:
track_fns = {
    "Linear Anchor": lambda: model(test_in),
    "SwiGLU 8/3 FFN": lambda: model.swiglu(weights["x_gate"]),
    "GeGLU 8/3 FFN": lambda: model.geglu(weights["x_gate"]),
    "Periodic Snake": lambda: model.snake(weights["x_snake"]),
    "Self-Normalizing SELU": lambda: model.selu(weights["x_selu"]),
    "LeakyReLU": lambda: F.leaky_relu(weights["x_dead"], 0.1),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Failure Mode: Dying ReLU Neuron Death vs Smooth Non-Vanishing Gradient Flow
x_neg = torch.tensor([-5.0, -3.0, -2.0, -1.0], requires_grad=True, device=device)
out_relu = F.relu(x_neg)
out_relu.sum().backward()
dead_grad_count = int(torch.sum(x_neg.grad == 0.0).item())
assert dead_grad_count == 4, f"Expected 4 dead gradients under ReLU, got {dead_grad_count}"

# Contrast with LeakyReLU maintaining active non-zero gradients everywhere:
x_leaky = torch.tensor([-5.0, -3.0, -2.0, -1.0], requires_grad=True, device=device)
out_leaky = F.leaky_relu(x_leaky, negative_slope=0.1)
out_leaky.sum().backward()
alive_grad_count = int(torch.sum(x_leaky.grad > 0.0).item())
assert alive_grad_count == 4, f"Expected 4 alive gradients under LeakyReLU, got {alive_grad_count}"
print(f"[FAILURE DEMONSTRATED] Dying ReLU produced {dead_grad_count}/4 dead units vs LeakyReLU {alive_grad_count}/4 alive!")

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), "Compiled model mismatch"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.")


### Key Takeaways & Algorithmic Summary

| Family | Primary Representative | Structural Strength | Pathological Vulnerability |
| :--- | :--- | :--- | :--- |
| **Piecewise Linear** | ReLU / LeakyReLU | Zero saturation on $\mathbb{R}^+$, fast computation | Negative saturation causes irreversible Dying ReLU. |
| **Smooth Gated** | SwiGLU / GeGLU | Bilinear gating, non-monotonic gradient flow | Requires $3\times$ linear projection matrices. |
| **Periodic** | Snake ($x + \frac{1}{a}\sin^2(ax)$) | Preserves frequency inductive bias in audio/PDEs | Hyperparameter sensitivity in frequency scale $a$. |
| **Self-Normalizing** | SELU | Preserves $(\mu=0, \sigma^2=1)$ across depth | Invalidated by BatchNorm or standard Dropout. |

### Curriculum Navigation & Synchronized Tracks
- [Track 1: Architectural Foundations (Markdown)](../../markdown/03_building_blocks_and_losses/01_activation_functions_zoo.md)
- [Track 2: Production PyTorch 2.x Module](../../pytorch2/03_building_blocks_and_losses/01_activation_functions_zoo.py)
- [Track 3: Functional JAX Module](../../jax/03_building_blocks_and_losses/01_activation_functions_zoo.py)
- [Track 4: Native Rust Candle Module](../../candle/03_building_blocks_and_losses/01_activation_functions_zoo.rs)
- [Track 5: Dual-Engine Hardware Kernels](../../kernels/03_building_blocks_and_losses/01_activation_functions_zoo.rs)
